# Jour 3 — Comprendre les séquences, RNN, LSTM et GRU

## Objectifs

- reconnaître une donnée séquentielle et lire sa forme
- comprendre le rôle de l’état caché d’un réseau récurrent
- distinguer SimpleRNN, LSTM et GRU sans entrer dans des mathématiques avancées
- entraîner un LSTM sur une petite séquence synthétique

## Quand l’ordre des valeurs contient de l’information

Une ligne tabulaire classique décrit une observation par un ensemble de variables. Une séquence ajoute un ordre : le troisième élément arrive après le deuxième. Permuter les éléments peut donc changer le sens.

Exemples de séquences : mesures successives d’un capteur, mots d’une phrase, échantillons audio, positions successives d’un objet ou consommation relevée heure par heure.

![Vecteur, image et séquence possèdent des formes différentes](../ressources/illustrations/jour_03/04_vecteur_image_sequence.png)

Pour un réseau récurrent, la forme habituelle est `(batch, pas_de_temps, variables)`. Un lot de 32 séquences, comportant chacune 30 instants et 2 capteurs, possède donc la forme `(32, 30, 2)`.

## Observation, pas de temps et variable

- Une **observation** est ici une séquence complète.
- Un **pas de temps** est une position dans cette séquence.
- Une **variable**, ou *feature*, est une valeur mesurée à chaque pas.
- La **longueur de séquence** est le nombre de pas observés.

Il ne faut pas confondre les 30 pas d’une même séquence avec un batch de 30 séquences. Le batch regroupe plusieurs observations ; l’axe temporel décrit l’ordre à l’intérieur de chacune.

## Le principe d’un réseau récurrent

![RNN déplié dans le temps](../ressources/illustrations/jour_03/04_rnn_deplie.png)

Un réseau récurrent lit les éléments l’un après l’autre. À chaque pas, il combine la nouvelle entrée avec un **état caché** transmis depuis le pas précédent. Cet état est un résumé numérique de ce qui a déjà été lu.

La même cellule et les mêmes poids sont réutilisés à tous les pas. On parle de réseau « récurrent » parce que l’état produit revient dans le calcul suivant. Le dessin déplie ce calcul dans le temps, mais il représente une seule cellule réutilisée.

Une écriture simplifiée est `nouvel état = activation(entrée actuelle + état précédent)`. Les matrices de poids et le biais sont volontairement omis ici : l’idée essentielle est la présence d’une mémoire transmise.

## Pourquoi LSTM et GRU existent-ils ?

Un `SimpleRNN` peut avoir du mal à conserver une information ancienne lorsque la séquence devient longue. Pendant la rétropropagation à travers de nombreux pas, le signal peut devenir extrêmement petit ou instable.

Les **LSTM** et **GRU** ajoutent des portes apprises. Une porte est un mécanisme qui décide quelle information conserver, oublier ou transmettre.

| Couche | Idée | Usage pédagogique |
|---|---|---|
| `SimpleRNN` | état caché simple | comprendre le principe de récurrence |
| `GRU` | mémoire contrôlée avec moins de portes | compromis plus compact |
| `LSTM` | état caché et mémoire contrôlés par plusieurs portes | dépendances plus longues |

Un LSTM n’enregistre pas des souvenirs sous forme de phrases. Sa mémoire est un vecteur de nombres appris pour réduire la loss.

## Différentes formes de problèmes séquentiels

- **plusieurs entrées → une sortie** : classer une phrase ou une séquence de capteur ;
- **plusieurs entrées → plusieurs sorties** : attribuer une classe à chaque instant ;
- **séquence passée → valeur future** : prévoir la prochaine mesure ;
- **séquence passée → séquence future** : prévoir plusieurs pas.

Dans Keras, `return_sequences=False` renvoie seulement l’état final de la couche récurrente. `return_sequences=True` renvoie une sortie à chaque pas. Le bon choix dépend donc de la forme de la cible.

Des séquences de longueurs différentes sont souvent complétées avec du **padding**. Un masque peut signaler au réseau quelles positions ont été ajoutées artificiellement et doivent être ignorées.

## Une expérience contrôlée : reconnaître une tendance

Nous créons des séquences synthétiques représentant une grandeur qui monte ou descend avec du bruit. Le label vaut `1` pour une tendance montante et `0` pour une tendance descendante.

Chaque observation contient 30 pas et une variable. Ce problème est volontairement simple : son but est de rendre visibles la forme des données et la circulation dans un LSTM, pas de battre un record de performance.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split


tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass
plt.style.use("seaborn-v0_8-whitegrid")

rng = np.random.default_rng(42)
n_sequences = 1200
steps = 30
time_axis = np.linspace(0, 1, steps, dtype="float32")
labels = rng.integers(0, 2, size=n_sequences).astype("int64")
directions = np.where(labels == 1, 1.0, -1.0).astype("float32")
starts = rng.uniform(-0.5, 0.5, size=n_sequences).astype("float32")
noise = rng.normal(0, 0.12, size=(n_sequences, steps)).astype("float32")

sequences = starts[:, None] + directions[:, None] * time_axis[None, :] + noise
sequences = sequences[..., np.newaxis].astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    sequences, labels, test_size=0.25, stratify=labels, random_state=42
)

print("Toutes les séquences :", sequences.shape)
print("Train :", X_train.shape, "Test :", X_test.shape)

fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
for ax, index in zip(axes, [0, 1, 2, 3]):
    ax.plot(sequences[index, :, 0])
    ax.set_title("montante" if labels[index] == 1 else "descendante")
    ax.set_xlabel("pas de temps")
axes[0].set_ylabel("valeur")
plt.tight_layout()
plt.show()

## Construire le LSTM

`Input(shape=(steps, 1))` signifie qu’une observation contient `steps` instants et une variable par instant. La dimension du batch n’est pas écrite : Keras l’ajoute lorsqu’il regroupe les observations.

`LSTM(16)` produit ici un vecteur final de 16 nombres. `Dense(1, activation="sigmoid")` transforme ce résumé en une probabilité comprise entre 0 et 1. La binary cross-entropy convient parce que la cible possède deux classes.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(steps, 1)),
    tf.keras.layers.LSTM(16),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_split=0.20,
    epochs=25,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)
model.summary()

## Vérifier l’apprentissage sans confondre les axes

Les courbes train et validation servent à vérifier que le modèle progresse et généralise. Le test n’est utilisé qu’après les choix d’entraînement.

Une probabilité supérieure ou égale à 0,5 devient ici la classe `1`. Ce seuil appartient à la décision ; le réseau produit d’abord une probabilité continue.

In [ ]:
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
for ax in axes:
    ax.set_xlabel("époque")
plt.tight_layout()
plt.show()

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
test_probabilities = model.predict(X_test, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.5).astype("int64")
print("Époques effectuées :", len(history_df))
print("Accuracy test :", round(test_accuracy, 3))

for index in range(5):
    print(
        f"exemple {index} : probabilité montante={test_probabilities[index]:.3f}",
        f"prédiction={test_predictions[index]}",
        f"réponse={y_test[index]}",
    )

## Ce que cette démonstration prouve — et ne prouve pas

Elle montre comment représenter un lot de séquences et entraîner un LSTM de bout en bout. Elle ne prouve pas qu’un LSTM est toujours le meilleur modèle. Une tendance aussi simple peut aussi être détectée par une règle, une régression ou un réseau Dense.

Dans un vrai projet, on commence donc par une baseline simple. Les modèles récurrents deviennent utiles lorsque l’ordre, le contexte ou des dépendances entre plusieurs pas apportent réellement de l’information.

Les Transformers traitent également des séquences, mais utilisent notamment un mécanisme d’attention plutôt qu’un état transmis strictement pas à pas. Ils constituent une autre famille, pas un nouveau nom pour les RNN.

## À vous de jouer — construire une séquence montante

Créez une séquence parfaitement croissante de 30 valeurs allant de `-0.8` à `0.8`. Donnez-lui la forme attendue par le modèle, obtenez sa probabilité et transformez celle-ci en classe avec le seuil `0.5`.

La forme attendue doit être `(1, 30, 1)` : un exemple, trente pas, une variable.

In [ ]:
# 1. Créez 30 valeurs croissantes avec np.linspace.
# 2. Donnez au tableau la forme (1, steps, 1).
# 3. Utilisez model.predict(..., verbose=0).
# 4. Affichez la probabilité et la classe obtenue avec un seuil de 0.5.
pass

## À retenir

Un réseau récurrent traite une séquence dans son ordre et transporte un état entre les pas. LSTM et GRU contrôlent mieux cette mémoire que le RNN simple. Avant de choisir l’une de ces couches, il faut identifier clairement les axes batch, temps et variables, ainsi que la forme de la sortie attendue.